# Correct DST k=10: patient-paired statistical analysis

This notebook rebuilds the five-model comparison with the correct GLAM-embedding DST-Prototype k=10. The primary inference is paired on the same independent test patients; image-level results retain patient clustering.

## Analysis contract

- Correct DST source: `G:\\611\\glam\\proto_embedding_rerun\\DST_k_10`
- Primary unit: patient; patient score = mean image probability, patient label = maximum image label.
- Primary test: paired DeLong, four prespecified DST-vs-baseline comparisons, Holm correction.
- Uncertainty: 20,000 paired stratified bootstraps; robustness: 20,000 paired score-swap permutations.
- Image sensitivity analysis: stratified patient-cluster bootstrap and patient-cluster score swap.
- Seed: `20260728`; all tests are two-sided with alpha 0.05.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, Markdown, display

OUTPUT_DIR = Path.cwd()
from analysis import run_analysis
results = run_analysis()

## Forced data and source validation

In [ ]:
display(pd.DataFrame([results['validation']]).T.rename(columns={0: 'observed'}))
display(results['source_audit'])

The analysis aborts unless the common cohort is exactly 8,409 images, 862 patients, and 19 positive patients; label conflicts, duplicate keys, missing predictions, and non-finite predictions must all equal zero. It also aborts unless the correct DST patient AUROC reproduces 0.864831 on all 8,433 images and 0.864581 on the common cohort.

## AUROC, AUPRC, confidence intervals, and descriptive threshold metrics

In [ ]:
cols = ['grain','model','n','positives','auc','auc_ci95_low','auc_ci95_high','auprc','auprc_ci95_low','auprc_ci95_high','sensitivity_0p5','specificity_0p5','bacc_0p5','f1_0p5']
display(results['model_metrics'][cols].round(6))

## Patient-level primary paired inference

In [ ]:
display(results['patient_tests'].round(6))

DeLong p values with Holm correction define the primary significance conclusions. The paired score-swap permutation is a robustness check for the small positive-patient count, not a replacement primary test.

## Image-level patient-cluster sensitivity analysis

In [ ]:
display(results['image_tests'].round(6))

## Patient-level ROC curves

In [ ]:
display(Image(filename=str(OUTPUT_DIR / 'patient_roc.png')))

## Paired ΔAUROC forest plot

In [ ]:
display(Image(filename=str(OUTPUT_DIR / 'patient_delta_auc_forest.png')))

## Paper-ready Statistical Analysis

In [ ]:
display(Markdown((OUTPUT_DIR / 'statistical_analysis_methods_en.md').read_text(encoding='utf-8')))

## Paper-ready Results

In [ ]:
display(Markdown((OUTPUT_DIR / 'results_en.md').read_text(encoding='utf-8')))

## 中文结论

In [ ]:
display(Markdown((OUTPUT_DIR / 'summary_zh.md').read_text(encoding='utf-8')))

## Reproducibility notes

The companion `analysis.py` contains the full source-loading, common-key construction, DeLong, bootstrap, permutation, Holm correction, plotting, and output code. Bootstrap sampling is stratified by positive/negative patient; identical resampling weights are used for every model. No model is retrained, and no result is paired across different training folds.